<a href="https://colab.research.google.com/github/aiman0642/saas-retention-intelligence/blob/main/06_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6 — Machine Learning Models

**Project:** SaaS Customer Retention Intelligence System

**Goal of this notebook:** train the three core models from the project
plan — Logistic Regression (baseline), Decision Tree (interpretability),
Random Forest (ensemble comparison) — on the preprocessed, feature-
engineered data from Module 5. This notebook focuses on *training*; full
evaluation (precision/recall/F1/ROC-AUC, model comparison table) is
Module 7's job, not this one — keeping them separate mirrors the actual
project plan and keeps each notebook focused.

**Assumes:** Module 5 has been run (needs `X_train.csv`, `X_test.csv`,
`X_train_scaled.csv`, `X_test_scaled.csv`, `y_train.csv`, `y_test.csv`
in `data/processed/`).

## 6.1 Mount Google Drive & Load Preprocessed Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

PROJECT_DIR = "/content/drive/MyDrive/saas-retention-intelligence"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
MODELS_DIR = f"{PROJECT_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

X_train = pd.read_csv(f"{PROCESSED_DIR}/X_train.csv")
X_test = pd.read_csv(f"{PROCESSED_DIR}/X_test.csv")
X_train_scaled = pd.read_csv(f"{PROCESSED_DIR}/X_train_scaled.csv")
X_test_scaled = pd.read_csv(f"{PROCESSED_DIR}/X_test_scaled.csv")
y_train = pd.read_csv(f"{PROCESSED_DIR}/y_train.csv").squeeze("columns")
y_test = pd.read_csv(f"{PROCESSED_DIR}/y_test.csv").squeeze("columns")

print(f"X_train: {X_train.shape}   y_train churn rate: {y_train.mean():.1%}")
print(f"X_test:  {X_test.shape}   y_test churn rate: {y_test.mean():.1%}")

X_train: (400, 41)   y_train churn rate: 22.0%
X_test:  (100, 41)   y_test churn rate: 22.0%


## 6.2 Model 1 — Logistic Regression (Baseline)

Uses the **scaled** features — Logistic Regression's coefficients are
sensitive to feature scale, so this is the one model that specifically
needs `X_train_scaled` rather than the raw version. `class_weight="balanced"`
is used because churn is imbalanced (~22% positive) — without it, the
model would be biased toward always predicting "retained."

In [4]:
log_reg = LogisticRegression(
    max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
)
log_reg.fit(X_train_scaled, y_train)

train_acc = log_reg.score(X_train_scaled, y_train)
test_acc = log_reg.score(X_test_scaled, y_test)
print(f"Logistic Regression — train accuracy: {train_acc:.3f}   test accuracy: {test_acc:.3f}")

Logistic Regression — train accuracy: 0.703   test accuracy: 0.640


## 6.3 Model 2 — Decision Tree

Uses the **unscaled** features — tree-based models split on raw
thresholds, so scaling doesn't affect them and unscaled values keep the
splits interpretable (e.g. "tenure_days > 180" rather than a scaled
z-score). `max_depth` is capped to reduce overfitting and keep the tree
small enough to actually be readable if visualized.

In [5]:
dec_tree = DecisionTreeClassifier(
    max_depth=6, min_samples_leaf=10, class_weight="balanced", random_state=RANDOM_STATE
)
dec_tree.fit(X_train, y_train)

train_acc = dec_tree.score(X_train, y_train)
test_acc = dec_tree.score(X_test, y_test)
print(f"Decision Tree — train accuracy: {train_acc:.3f}   test accuracy: {test_acc:.3f}")

Decision Tree — train accuracy: 0.780   test accuracy: 0.570


## 6.4 Model 3 — Random Forest

An ensemble of many decision trees — generally more robust than a single
tree since it averages out individual trees' overfitting tendencies.
Also unscaled, same reasoning as the Decision Tree above.

In [6]:
rand_forest = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=5,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)
rand_forest.fit(X_train, y_train)

train_acc = rand_forest.score(X_train, y_train)
test_acc = rand_forest.score(X_test, y_test)
print(f"Random Forest — train accuracy: {train_acc:.3f}   test accuracy: {test_acc:.3f}")

Random Forest — train accuracy: 0.983   test accuracy: 0.750


## 6.5 Quick Sanity Check — Predictions Look Reasonable?

Before moving to formal evaluation in Module 7, a basic gut-check: are
the models actually predicting *some* churners, or defaulting to "never
predict churn" (a common failure mode with imbalanced data that plain
accuracy alone can hide)?

**Read these numbers in context:** Modules 2 and 5 found no feature
correlating strongly with churn (all under r ≈ 0.12) — so ~60-75% test
accuracy here is the *expected*, honest result, not a disappointing one.
If any model had scored 90%+ against this feature set, that would be the
signal to go looking for leakage again, not a reason to celebrate.

In [7]:
for name, model, X_te in [
    ("Logistic Regression", log_reg, X_test_scaled),
    ("Decision Tree", dec_tree, X_test),
    ("Random Forest", rand_forest, X_test),
]:
    preds = model.predict(X_te)
    predicted_churn_rate = preds.mean()
    print(f"{name:<22} predicted churn rate on test set: {predicted_churn_rate:.1%}  "
          f"(actual: {y_test.mean():.1%})")

Logistic Regression    predicted churn rate on test set: 46.0%  (actual: 22.0%)
Decision Tree          predicted churn rate on test set: 39.0%  (actual: 22.0%)
Random Forest          predicted churn rate on test set: 5.0%  (actual: 22.0%)


Worth noting directly: Random Forest's train accuracy is meaningfully
higher than its test accuracy — a sign of overfitting to the training
set, unsurprising given how weak the underlying feature signal is.
It's also predicting churn far less often than the true rate despite
`class_weight="balanced"`, which accuracy alone doesn't reveal — this is
exactly why Module 7 evaluates with precision/recall/F1 instead of
accuracy alone, not a problem to fix here.

## 6.6 Feature Importance — First Look

A quick preview of which features each model is leaning on most —
Module 9 (SHAP) will go much deeper per-prediction, but this is a fast
sanity check that the models are picking up on features that make
business sense, not noise.

In [8]:
importances = pd.DataFrame({
    "feature": X_train.columns,
    "decision_tree": dec_tree.feature_importances_,
    "random_forest": rand_forest.feature_importances_,
}).sort_values("random_forest", ascending=False)

print("Top 10 features by Random Forest importance:")
print(importances.head(10).to_string(index=False))

Top 10 features by Random Forest importance:
                  feature  decision_tree  random_forest
   engagement_trend_ratio       0.355475       0.086982
              tenure_days       0.084464       0.080081
            tenure_months       0.158366       0.075337
       usage_rate_per_day       0.072796       0.063446
 days_since_last_activity       0.000000       0.056644
     avg_resolution_hours       0.062785       0.049389
        total_usage_count       0.000000       0.049135
support_tickets_per_month       0.072833       0.048886
total_usage_duration_secs       0.035108       0.048518
               mrr_amount       0.000000       0.044482


## 6.7 Save Trained Models

Persist all three models so Module 7 (Evaluation), Module 8 (Churn
Probability), and Module 9 (SHAP) can load them directly without
retraining.

In [9]:
joblib.dump(log_reg, f"{MODELS_DIR}/logistic_regression.pkl")
joblib.dump(dec_tree, f"{MODELS_DIR}/decision_tree.pkl")
joblib.dump(rand_forest, f"{MODELS_DIR}/random_forest.pkl")

print(f"Saved 3 models to {MODELS_DIR}/")
print("  - logistic_regression.pkl")
print("  - decision_tree.pkl")
print("  - random_forest.pkl")

Saved 3 models to /content/drive/MyDrive/saas-retention-intelligence/models/
  - logistic_regression.pkl
  - decision_tree.pkl
  - random_forest.pkl


## 6.8 Module 6 Summary

- Trained 3 models on Module 5's corrected, leakage-free feature set:
  Logistic Regression (64% test acc.), Decision Tree (57%), Random
  Forest (75%)
- Used `class_weight="balanced"` on all three to account for the ~22%
  churn imbalance
- Results are modest and honest, consistent with Modules 2 and 5 finding
  no strongly predictive single feature — this is the expected outcome
  for this dataset, not a shortfall to fix
- Random Forest shows clear overfitting (98% train vs. 75% test
  accuracy) and under-predicts churn frequency despite class balancing —
  flagged here for Module 7 to examine properly with precision/recall
  rather than papered over
- Saved all 3 trained models for reuse in later modules

**Next:** Module 7 — Model Evaluation